# VoxDoc - Google Colab Deployment

Deploy the VoxDoc Voice Symptom Intake & Documentation Assistant on Google Colab with GPU support.

## Prerequisites
- **Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU)
- **Hugging Face token** with "Read" access ([get one here](https://huggingface.co/settings/tokens))
- Accept model terms for [google/medasr](https://huggingface.co/google/medasr) and [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
- **ngrok authtoken** ([get one here](https://dashboard.ngrok.com/get-started/your-authtoken))

## What you get
- MedGemma SOAP documentation generation on a free T4 GPU
- MedASR + Whisper speech-to-text
- Medical NER (condition & medication extraction)
- Public URL via ngrok tunnel

## Step 1: Check GPU & Clone Repository

In [ ]:
import torch, subprocess

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB VRAM)")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> Select 'T4 GPU'")
    print("Then re-run this cell.")

# Clone repository (colab-gpu-testing branch)
!rm -rf /content/voice-symptom-documentation-assistant
!git clone --branch colab-gpu-testing https://github.com/JoelJohnsonThomas/voice-symptom-documentation-assistant.git /content/voice-symptom-documentation-assistant
%cd /content/voice-symptom-documentation-assistant
print(f"\nRepo cloned (branch: colab-gpu-testing). Working directory: /content/voice-symptom-documentation-assistant")

## Step 2: Install All Dependencies

This takes ~3-5 minutes. Every package the app needs is installed here — no manual pip installs later.

In [ ]:
import subprocess, sys

def pip(*args):
    """Install packages and show result."""
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  FAILED: pip install {' '.join(args)}")
        if r.stderr:
            print(r.stderr[-1500:])
        return False
    return True

print("1/8  Transformers (MedGemma-compatible commit)...")
pip("git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5")

print("2/8  Web framework & networking...")
pip("fastapi", "uvicorn[standard]", "python-multipart",
    "websockets", "httpx")

print("3/8  ML & audio...")
pip("accelerate", "huggingface_hub>=0.24.0")
pip("librosa", "soundfile", "audioread", "noisereduce")
pip("openai-whisper", "--no-deps")  # --no-deps keeps Colab's torch untouched
pip("more-itertools", "tiktoken", "numba")  # whisper's actual deps, without torch
pip("faster-whisper")
pip("bitsandbytes")  # optional quantization support

print("4/8  Config & database...")
pip("pydantic", "pydantic-settings", "python-dotenv")
pip("sqlalchemy[asyncio]", "aiosqlite", "alembic")

print("5/8  Auth & security...")
pip("PyJWT", "passlib[bcrypt]", "pyotp",
    "qrcode[pil]", "authlib")
pip("cryptography")

print("6/8  Medical NLP (spaCy + optional SciSpaCy NER models)...")
pip("spacy", "--prefer-binary")
# scispacy pins an old spaCy with no wheels for Colab's Python, so skip the package.
# Models install with --no-deps; NER is optional and the app runs without it.
for url in ["https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz",
            "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz"]:
    if not pip(url, "--no-deps"):
        print("  (optional) NER model skipped — app will run without medical NER")
pip("presidio-analyzer", "presidio-anonymizer")

print("7/8  RAG & knowledge base...")
pip("chromadb", "sentence-transformers")

print("8/8  Infrastructure & utilities...")
pip("pyngrok", "nest-asyncio")

# Check CUDA in a fresh process — this kernel may hold a stale torch import.
CUDA_CHECK = "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda)"
def cuda_ok():
    r = subprocess.run([sys.executable, "-c", CUDA_CHECK], capture_output=True, text=True)
    return r.returncode == 0, (r.stdout or r.stderr).strip().splitlines()[-1:]

ok, info = cuda_ok()
if not ok:
    print("\n  torch can't use the GPU — reinstalling the standard CUDA build from PyPI...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torch", "torchaudio", "torchvision"])
    pip("torch", "torchaudio", "torchvision")
    ok, info = cuda_ok()
print(f"\nGPU torch: {'OK' if ok else 'BROKEN'} {info}")

# Verify critical imports (fresh process, so results reflect what uvicorn will see)
print("\nVerifying critical imports...")
mods = ["fastapi", "uvicorn", "torch", "transformers", "whisper", "librosa", "sqlalchemy",
        "aiosqlite", "alembic", "jwt", "passlib", "pyotp", "qrcode", "authlib", "cryptography",
        "spacy", "chromadb", "sentence_transformers", "httpx", "pyngrok", "presidio_analyzer",
        "soundfile", "pydantic", "pydantic_settings"]
check = "import importlib,sys\nbad=[]\nfor m in sys.argv[1:]:\n    try: importlib.import_module(m)\n    except Exception as e: bad.append(f'{m}: {e}')\nprint('\\n'.join(bad))"
r = subprocess.run([sys.executable, "-c", check, *mods], capture_output=True, text=True)
missing = r.stdout.strip()
if missing:
    print("Some packages failed to import:\n" + missing)
else:
    print("All packages installed successfully!")
print("\nIMPORTANT: Runtime -> Restart session now, then continue with Step 3.")

## Step 3: Configure Environment

Replace `HF_TOKEN` and `NGROK_AUTH_TOKEN` with your own values.

In [ ]:
import os
os.chdir("/content/voice-symptom-documentation-assistant")

# ──────────────────────────────────────────────
#  REPLACE THESE TWO VALUES WITH YOUR OWN
# ──────────────────────────────────────────────
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
# ──────────────────────────────────────────────

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

    # Models — whisper-small for high accuracy speech recognition
    f.write("MEDASR_MODEL=google/medasr\n")
    f.write("MEDGEMMA_MODEL=google/medgemma-1.5-4b-it\n")
    f.write("WHISPER_MODEL=openai/whisper-small\n")
    f.write("MODEL_CACHE_DIR=./models\n")

    # GPU — MedGemma runs locally on T4
    f.write("DEVICE=cuda\n")
    f.write("ENABLE_GPU=true\n")
    f.write("COLAB_MODE=true\n")

    # MedGemma provider — local means load on this GPU
    f.write("MEDGEMMA_PROVIDER=local\n")
    f.write("HF_INFERENCE_PROVIDER=\n")
    f.write("HF_INFERENCE_TIMEOUT=120\n")

    # Generation parameters
    f.write("MEDGEMMA_MAX_TOKENS=1024\n")
    f.write("MEDGEMMA_REPETITION_PENALTY=1.1\n")

    # Compliance — REQUIRED or MedGemma won't load
    f.write("MEDGEMMA_TERMS_ACKNOWLEDGED=true\n")
    f.write("ENFORCE_MEDGEMMA_TERMS_ACKNOWLEDGEMENT=true\n")
    f.write("ALLOW_PHI_LOGGING=false\n")
    f.write("ENABLE_PHI_PERSISTENCE=false\n")
    f.write("AUDIT_LOGGING_ENABLED=true\n")

    # Audio — 2s streaming interval (fast on GPU)
    f.write("AUDIO_SAMPLE_RATE=16000\n")
    f.write("MAX_AUDIO_DURATION_SECONDS=300\n")
    f.write("STREAMING_INTERVAL_SECONDS=2.0\n")

    # Rate limiting
    f.write("RATE_LIMITING_ENABLED=true\n")
    f.write("RATE_LIMIT_GENERAL_RPM=60\n")
    f.write("RATE_LIMIT_INFERENCE_RPM=10\n")
    f.write("QUEUE_MAX_CONCURRENT_INFERENCES=2\n")
    f.write("QUEUE_MAX_SIZE=20\n")
    f.write("QUEUE_TIMEOUT_SECONDS=120\n")
    f.write("QUEUE_ESTIMATED_INFERENCE_SECONDS=10\n")

    # Monitoring
    f.write("METRICS_ENABLED=true\n")
    f.write("STRUCTURED_LOGGING_ENABLED=true\n")
    f.write("METRICS_ENDPOINT_AUTH_REQUIRED=false\n")

    # Security (dev defaults — not for production)
    f.write("ENCRYPTION_AT_REST_ENABLED=false\n")
    f.write("ENCRYPTION_MASTER_KEY=CHANGE_ME_IN_PRODUCTION\n")
    f.write("JWT_SECRET_KEY=CHANGE_ME_IN_PRODUCTION\n")

    # RAG (disabled for faster startup)
    f.write("RAG_ENABLED=false\n")
    f.write("RAG_EMBEDDING_MODEL=NeuML/pubmedbert-base-embeddings\n")

    # Logging
    f.write("LOG_LEVEL=INFO\n")

# Configure ngrok
from pyngrok import ngrok
import nest_asyncio
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

print("Environment configured!")
print(f"  Whisper model: openai/whisper-small (high accuracy)")
print(f"  MedGemma: local GPU inference (google/medgemma-1.5-4b-it)")
print(f"  MEDGEMMA_TERMS_ACKNOWLEDGED: true")
print(f"  Streaming interval: 2.0s (fast on GPU)")
print(f"  ngrok: ready")

## Step 4: Start the Server

This will:
1. Load MedASR model on GPU (~30 seconds)
2. Load MedGemma model on GPU (~2-3 minutes, first time downloads ~8 GB)
3. Start the FastAPI server on port 8000
4. Create a public ngrok URL

The ngrok URL will be printed at the top — open it in your browser.

In [ ]:
import os
os.chdir("/content/voice-symptom-documentation-assistant")
os.environ["MEDGEMMA_TERMS_ACKNOWLEDGED"] = "true"

from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  PUBLIC URL: {public_url}")
print(f"{'='*60}")
print("  Open this URL in your browser to access VoxDoc!")
print(f"{'='*60}\n")

# Start the server (this blocks until you stop the cell)
!cd /content/voice-symptom-documentation-assistant && python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

## Usage & Troubleshooting

### How to use
1. Copy the ngrok URL printed above and open it in your browser
2. Click **Start Recording** or upload an audio file
3. The app transcribes speech and generates SOAP documentation using MedGemma

### To stop
- Click the **Stop** button on the cell above, or press **Ctrl+C**

### Common issues

| Problem | Fix |
|---------|-----|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU, then restart |
| `ModuleNotFoundError` | Re-run the Step 2 install cell |
| `Skipping MedGemma preload: terms not acknowledged` | Make sure `.env` has `MEDGEMMA_TERMS_ACKNOWLEDGED=true` |
| `torch.cuda.is_available() is False` | CUDA torch got overwritten — re-run Step 2 |
| ngrok `ERR_NGROK_108` | Free tier allows 1 tunnel — stop other Colab sessions |
| `en_ner_bc5cdr_md not found` | Re-run Step 2 (installs SciSpaCy models) |
| Documentation shows "N/A" | Model may need warmup — try generating again |

### Notes
- Free Colab sessions last up to 12 hours
- The ngrok URL changes each restart
- Models are cached after first download (faster restarts)
- GPU inference is ~10x faster than CPU